# §4.4  Acceptance Criteria

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../../../src').resolve()))
from config import get_snapshot_redshift

try:
    from utils.matplotlib_config import setconfig
    setconfig()
except ImportError:
    pass

In [ ]:
DATA_ROOT = Path('../../../../data/2pcf/scope_xi')
MODEL     = 'lc16'
SIM       = 'L800'
N_REF     = 1024   # full-box Corrfunc reference (pending — falls back to max available n)

# Quick-look config (sections 1–2)
QK_IZ        = 155
QK_MSTAR_TAG = 'mstar9.0'
QK_Z         = get_snapshot_redshift(f'iz{QK_IZ}', 'L800')

# n values to show in detail and headline plots
PLOT_N     = [2,4,8,16,32,64, 128, 256]#[2, 8, 32, 128, 256]
HEADLINE_N = [2,4,8,16,32,64, 128, 256]#[4, 16, 64, 256]
# n=128 and n=256 are infeasible for mstar_none: runtime >> cosma8 8h/16h limits
PLOT_N_MSTAR_NONE = [n for n in PLOT_N if n < 128]

print(f'Data root:   {DATA_ROOT}')
print(f'Quick-look:  iz{QK_IZ}  →  z = {QK_Z:.3f}  ({QK_MSTAR_TAG})')

In [ ]:
def load_and_aggregate(iz: int, mstar_tag: str) -> pd.DataFrame | None:
    """Load all scope_xi CSVs for one (iz, mstar_tag) and aggregate over seeds.

    The n=1024 Corrfunc full-box run is always used as the reference.  Residual
    columns (frac_diff_*, dlog_*) will be NaN until that reference is available.

    Returns a DataFrame with per-(n_subvol, bin_idx) statistics, or None if no data.
    """
    base = DATA_ROOT / MODEL / f'iz{iz}' / mstar_tag
    if not base.is_dir():
        return None

    files = sorted(base.glob(f'n*/seed*/scope_xi_{SIM}_iz{iz}.csv'))
    if not files:
        return None

    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df = df.drop_duplicates(subset=['selection_seed', 'n_subvol', 'bin_idx'])

    if N_REF not in df['n_subvol'].values:
        print(f'  WARNING: n={N_REF} reference not yet available for iz{iz} {mstar_tag} — residuals will be empty')

    # Compute statistics *only* over strictly positive ξ values when forming spreads/counts.
    # This avoids including zero or negative ξ in error/spread calculations.
    g = (df
         .groupby(['n_subvol', 'bin_idx', 'r_mid'], as_index=False)
         .agg(
             xi_corr_mean   = ('xi_corrected', lambda x: next((val for val in x if val != 0), np.nan)) ,#('xi_corrected', 'mean'),
             xi_corr_std    = ('xi_corrected', lambda x: float(np.nanstd(x[x > 0])) if np.any(x > 0) else np.nan),
             xi_naive_mean  = ('xi_naive',lambda x: next((val for val in x if val != 0), np.nan)), #('xi_naive',     'mean'),
             xi_naive_std   = ('xi_naive',     lambda x: float(np.nanstd(x[x > 0])) if np.any(x > 0) else np.nan),
             xi_corr_p16    = ('xi_corrected', lambda x: float(np.nanpercentile(x[x > 0], 16)) if np.any(x > 0) else np.nan),
             xi_corr_p84    = ('xi_corrected', lambda x: float(np.nanpercentile(x[x > 0], 84)) if np.any(x > 0) else np.nan),
             n_seeds_corr   = ('xi_corrected', lambda x: int(np.count_nonzero(x > 0))),
             n_seeds_naive  = ('xi_naive',     lambda x: int(np.count_nonzero(x > 0))),
         ))

    # Error on the mean using only the positive-value counts; leave NaN where no positive samples.
    g['xi_corr_err']  = np.where(g['n_seeds_corr'] > 0, g['xi_corr_std']  / np.sqrt(g['n_seeds_corr']),  np.nan)
    g['xi_naive_err'] = np.where(g['n_seeds_naive'] > 0, g['xi_naive_std'] / np.sqrt(g['n_seeds_naive']), np.nan)

    # Merge reference values using bin_idx (not r_mid, which can differ by rounding)
    ref = (g[g['n_subvol'] == N_REF][['bin_idx', 'xi_corr_mean', 'r_mid']]
             .rename(columns={'xi_corr_mean': 'xi_ref', 'r_mid': 'r_mid_ref'}))
    g = g.merge(ref, on='bin_idx', how='left')
    # Use reference r_mid (more accurate since it's from the full box)
    g['r_mid'] = g['r_mid_ref'].fillna(g['r_mid'])
    g = g.drop(columns=['r_mid_ref'])

    # Fractional residual: (xi - xi_ref) / |xi_ref|
    safe_ref = np.where(np.abs(g['xi_ref']) > 0, g['xi_ref'], np.nan)
    g['frac_diff_corr']  = (g['xi_corr_mean']  - g['xi_ref']) / np.abs(safe_ref)
    g['frac_diff_naive'] = (g['xi_naive_mean'] - g['xi_ref']) / np.abs(safe_ref)

    # log-ratio where both are positive (for threshold plot)
    pos_c = (g['xi_corr_mean'] > 0) & (g['xi_ref'] > 0)
    g['dlog_corr']  = np.where(pos_c, np.log10(g['xi_corr_mean'] / g['xi_ref']),  np.nan)
    pos_n = (g['xi_naive_mean'] > 0) & (g['xi_ref'] > 0)
    g['dlog_naive'] = np.where(pos_n, np.log10(g['xi_naive_mean'] / g['xi_ref']), np.nan)

    return g


def _n_colors(n_values):
    cmap = mpl.colormaps['plasma']
    return [cmap(0.1 + 0.75 * i / max(len(n_values) - 1, 1)) for i in range(len(n_values))]


def _mstar_label(mstar_tag: str) -> str:
    if mstar_tag == 'mstar_none':
        return 'no $M_*$ cut'
    val = mstar_tag.replace('mstar', '')
    return f'$M_*/M_\\odot h^{{-1}} > {val}$'


print('Helpers defined.')

def compute_convergence_metrics(g: pd.DataFrame) -> pd.DataFrame:
    """Compute multiple convergence metrics per N_subvol.

    Returns one row per n_subvol (excluding N_REF) with columns:
      med_dlog_{corr,naive}   — median |Δlog₁₀ξ|
      rms_frac_{corr,naive}   — RMS fractional error
      max_frac_{corr,naive}   — worst-case |frac error| over r bins
      f5_{corr,naive}         — fraction of r bins within 5% of reference
      f10_{corr,naive}        — fraction of r bins within 10% of reference
      med_scatter             — median seed scatter: (p84−p16)/(2|ξ_ref|)
    All metrics computed only over bins where ξ_ref > 0.
    """
    records = []
    for n in sorted(g['n_subvol'].unique()):
        if n == N_REF:
            continue
        sub  = g[(g['n_subvol'] == n) & (g['xi_ref'] > 0)].copy()
        if sub.empty:
            continue
        fc = sub['frac_diff_corr'].dropna().values
        fn = sub['frac_diff_naive'].dropna().values
        dc = sub['dlog_corr'].dropna().values
        dn = sub['dlog_naive'].dropna().values
        scatter = ((sub['xi_corr_p84'] - sub['xi_corr_p16']) /
                   (2 * sub['xi_ref'].abs())).dropna().values
        records.append({
            'n_subvol':       int(n),
            'med_dlog_corr':  np.nanmedian(np.abs(dc))                   if len(dc) else np.nan,
            'med_dlog_naive': np.nanmedian(np.abs(dn))                   if len(dn) else np.nan,
            'rms_frac_corr':  np.sqrt(np.nanmean(fc**2))                 if len(fc) else np.nan,
            'rms_frac_naive': np.sqrt(np.nanmean(fn**2))                 if len(fn) else np.nan,
            'max_frac_corr':  float(np.nanmax(np.abs(fc)))               if len(fc) else np.nan,
            'max_frac_naive': float(np.nanmax(np.abs(fn)))               if len(fn) else np.nan,
            'f5_corr':        float(np.mean(np.abs(fc) < 0.05))          if len(fc) else np.nan,
            'f5_naive':       float(np.mean(np.abs(fn) < 0.05))          if len(fn) else np.nan,
            'f10_corr':       float(np.mean(np.abs(fc) < 0.10))          if len(fc) else np.nan,
            'f10_naive':      float(np.mean(np.abs(fn) < 0.10))          if len(fn) else np.nan,
            'med_scatter':    float(np.nanmedian(scatter))                if len(scatter) else np.nan,
        })
    return pd.DataFrame(records)

## §4.4  Acceptance Criteria

### Formal criterion

**N\* = min m such that the two-halo median |Δlog₁₀ξ| < 0.05 dex** over r ∈ [5, 30] h⁻¹ Mpc.

- 5% threshold: (i) operationally relevant scale range for large-scale bias + BAO; (ii) ≈ 1σ contribution to clustering likelihood error budget; (iii) level at which SCOPE bias becomes sub-dominant to sample variance (§5.5)
- Naïve estimator: ≥ 0.06–0.23 dex bias at r=5–30 Mpc/h even at N=16–64; the correction is essential

### N\* by selection (from L800/lc16 data)

| Selection | iz155 (z≈1.5) | iz207 (z≈0.5) | iz271 (z=0.0) |
|-----------|--------------|--------------|---------------|
| mstar9.0  | 64           | 40           | 32            |
| mstar10.0 | 64           | 64           | 40            |
| mstar11.0 | > 256        | > 256        | 256           |

Two-halo median |Δlog₁₀ξ| at selected N (for reference):

| Selection | N=8  | N=16 | N=32 | N=64 | N=128 |
|-----------|------|------|------|------|-------|
| mstar9 z=0 | 0.112 | 0.064 | 0.039✓ | 0.020✓ | 0.013✓ |
| mstar9 z=1.5 | 0.238 | 0.112 | 0.068 | 0.036✓ | 0.018✓ |
| mstar10 z=0 | 0.151 | 0.087 | 0.052 | 0.025✓ | 0.016✓ |
| mstar10 z=1.5 | 0.257 | 0.160 | 0.074 | 0.045✓ | 0.025✓ |

✓ = below 0.05 dex threshold

### Secondary compliance check (f₅ in two-halo regime [5–30 Mpc/h] at N=64)

| Selection | iz155 (z≈1.5) | iz207 (z≈0.5) | iz271 (z=0.0) |
|-----------|--------------|--------------|---------------|
| mstar9.0  | 30%          | 44%          | 53%           |
| mstar10.0 | 26%          | 31%          | 42%           |

Note: low f₅ reflects the stringent per-bin 5% criterion at scales with large cosmic variance;
the median |Δlog₁₀ξ| criterion (N\*) is the primary metric.

### Practical recommendation

- **N = 64**: broad clustering analyses (power-spectrum level, HOD forward models, cross-correlations)
- **N = 128**: precision inference requiring per-bin accuracy at the few-per-cent level across all scales
- **mstar11.0**: SCOPE not recommended at N < 256; verify ≥1 pair per sub-volume at scales of interest
- **global N\*** (dominated by one-halo scales) is 4–16 and should not be confused with the two-halo N\*

The N\* values and detailed convergence metrics are computed numerically in the cell below.

In [ ]:
# Compute N* (two-halo median |dlog| < 0.05) and f5 at N=16 for all cells
R_2H_MIN, R_2H_MAX = 5.0, 30.0
THRESH = 0.05

print(f"{'iz':>6}  {'mstar_tag':>12}  {'N* (2h)':>10}  {'med|dlog| at N*':>18}  {'f5 at N=16':>12}")
print('-' * 70)
for iz in IZ_LIST:
    for mtag in MSTAR_TAGS:
        g = load_and_aggregate(iz, mtag)
        if g is None:
            print(f'{iz:>6}  {mtag:>12}  (no data)')
            continue
        g2h = g[(g['r_mid'] >= R_2H_MIN) & (g['r_mid'] <= R_2H_MAX)]
        ns  = sorted(n for n in g['n_subvol'].unique() if n != N_REF)
        nstar, nstar_med = None, np.nan
        for n in ns:
            sub = g2h[g2h['n_subvol'] == n]
            med = np.nanmedian(np.abs(sub['dlog_corr'].dropna()))
            if np.isfinite(med) and med < THRESH:
                nstar, nstar_med = n, med
                break
        # f5 at N=16
        if 16 in ns:
            sub16 = g[(g['n_subvol']==16) & (g['xi_corr_mean']>0) & (g['xi_ref']>0)].dropna(subset=['frac_diff_corr'])
            f5 = f"{(np.abs(sub16['frac_diff_corr']) < 0.05).mean():.0%}" if not sub16.empty else 'N/A'
        else:
            f5 = 'N/A'
        ns_str = f'{nstar}' if nstar else f'>{ ns[-1]}'
        print(f'{iz:>6}  {mtag:>12}  {ns_str:>10}  {nstar_med:>18.4f}  {f5:>12}')
